In [2]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
import torch.nn as nn
import numpy as np
from tqdm import tqdm,trange


In [3]:
import torch

def patchify(images, patch_h, patch_w):
    n, c, h, w = images.shape
    assert h % patch_h == 0, f"Image height {h} not divisible by patch height {patch_h}"
    assert w % patch_w == 0, f"Image width {w} not divisible by patch width {patch_w}"
    
    num_patches_h = h // patch_h
    num_patches_w = w // patch_w
    
    patches = images.reshape(n, c, num_patches_h, patch_h, num_patches_w, patch_w)
    
    patches = patches.permute(0, 2, 4, 1, 3, 5)
    

    patches = patches.reshape(n, num_patches_h * num_patches_w, -1)
    
    return patches


In [4]:
import torch
import math

def get_positional_embedding(sequence_length, d):

    positions = torch.arange(sequence_length, dtype=torch.float32).unsqueeze(1)
    
    div_term = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * -(math.log(10000.0) / d))
    
    results = torch.zeros(sequence_length, d)
    
    results[:, 0::2] = torch.sin(positions * div_term)
    
    results[:, 1::2] = torch.cos(positions * div_term)
    
    return results

In [5]:
class MyMsa(nn.Module):
    def __init__(self,d,n_heads=2):
        super(MyMsa,self).__init__()
        self.d=d
        self.n_heads=n_heads

        assert d%n_heads==0

        d_head=int(d/n_heads)
        self.q_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.k_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.v_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.d_head=d_head
        self.softmax=nn.Softmax(dim=-1)
        self.out_proj=nn.Linear(d,d)

    def forward(self,sequences):
        result=[]
        for sequence in sequences:
            seq_result=[]
            for head in range(self.n_heads):
                q_mapping=self.q_mappings[head]
                k_mapping=self.k_mappings[head]
                v_mapping=self.v_mappings[head]

                seq=sequence[:,head*self.d_head:(head+1)*self.d_head]
                q,k,v=q_mapping(seq),k_mapping(seq),v_mapping(seq)

                attention=self.softmax(q@k.T/(self.d_head**2))
                seq_result.append(attention@v)

            result.append(torch.hstack(seq_result))

        return self.out_proj(torch.cat([torch.unsqueeze(r,dim=0) for r in result]))
                
        

In [6]:
class MyViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMsa(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d)
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

In [7]:
class MyVit(nn.Module):
    def __init__(self, chw=(3,224,224), n_patches=14, hidden_d=768, n_blocks=12, n_heads=12, out_d=10):
        super(MyVit,self).__init__()
        self.chw=chw
        self.n_patches=n_patches
        self.n_blocks=n_blocks
        self.n_heads=n_heads
        self.hidden_d=hidden_d
        self.out_d=out_d
        
        # Calculate physical pixel size of the patch
        self.patch_h = int(chw[1] / n_patches)
        self.patch_w = int(chw[2] / n_patches)

        assert chw[1] % n_patches == 0
        assert chw[2] % n_patches == 0 
        
        # 1. Linear Mapper (Calculates exactly 768)
        self.input_d= int(chw[0] * self.patch_h * self.patch_w)      
        self.linear_mapper=nn.Linear(self.input_d, self.hidden_d)

        # 2. Class Token
        self.class_token=nn.Parameter(torch.rand(1, self.hidden_d))

        # 3. Positional Embedding (Fixed UserWarning)
        pos_data = get_positional_embedding(self.n_patches**2 + 1, self.hidden_d)
        self.pos_embedding = nn.Parameter(pos_data.clone().detach())

        # 4. Transformer Blocks
        self.blocks=nn.ModuleList([MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)])

        # 5. MLP (Not used for DPT, but left here to prevent breaking old code)
        self.mlp=nn.Sequential(
            nn.Linear(self.hidden_d, self.out_d),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, images):
        n, c, h, w = images.shape
        
        patches = patchify(images, self.patch_h, self.patch_w)
        
        tokens = self.linear_mapper(patches)
        
        # Add class token
        tokens = torch.stack([torch.vstack((self.class_token, tokens[i])) for i in range(len(tokens))])
        
        pos_embed = self.pos_embedding.repeat(n, 1, 1)
        out = tokens + pos_embed

        extracted_tokens = []
        for i, block in enumerate(self.blocks):
            out = block(out)
            if i in [2, 5, 8, 11]: 
                extracted_tokens.append(out)
                
        return extracted_tokens

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReassembleBlock(nn.Module):
    def __init__(self, in_channels, out_channels=256, spatial_size=14, scale_factor=1):
        super(ReassembleBlock, self).__init__()
        self.spatial_size = spatial_size
        
        self.project = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        
        if scale_factor > 1:
            self.resample = nn.ConvTranspose2d(
                out_channels, out_channels, 
                kernel_size=int(scale_factor*2), 
                stride=int(scale_factor), 
                padding=int(scale_factor//2)
            )
        elif scale_factor < 1:
            stride = int(1 / scale_factor)
            self.resample = nn.Conv2d(
                out_channels, out_channels, 
                kernel_size=3, stride=stride, padding=1
            )
        else:
            self.resample = nn.Identity()

    def forward(self, tokens):
        batch_size = tokens.shape[0]
        
        cls_token = tokens[:, 0:1, :]
        spatial_tokens = tokens[:, 1:, :]
        spatial_tokens = spatial_tokens + cls_token 
        
        grid = spatial_tokens.reshape(batch_size, self.spatial_size, self.spatial_size, -1)
        grid = grid.permute(0, 3, 1, 2) 
        
        out = self.project(grid)
        out = self.resample(out)
        
        return out

In [9]:
class ResidualConvUnit(nn.Module):
    def __init__(self, features):
        super(ResidualConvUnit, self).__init__()
        self.conv1 = nn.Conv2d(features, features, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(features, features, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(x)
        out = self.conv1(out)
        out = self.relu(out)
        out = self.conv2(out)
        return out + x 

class FeatureFusionBlock(nn.Module):
    def __init__(self, features=256):
        super(FeatureFusionBlock, self).__init__()
        self.resConfUnit1 = ResidualConvUnit(features)
        self.resConfUnit2 = ResidualConvUnit(features)
        
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.project = nn.Conv2d(features, features, kernel_size=3, padding=1)

    def forward(self, x, previous_stage_output=None):
        out = self.resConfUnit1(x)
        
        if previous_stage_output is not None:
            out = out + previous_stage_output
            
        out = self.resConfUnit2(out)
        out = self.upsample(out)
        out = self.project(out)
        
        return out

In [10]:
class DPT(nn.Module):
    def __init__(self, vit_encoder, embed_dim=768, features=256, spatial_size=14):
        super(DPT, self).__init__()
        self.encoder = vit_encoder
        
        
        self.reassemble_4 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=4)
        self.reassemble_8 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=2)
        self.reassemble_16 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=1)
        self.reassemble_32 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=0.5)
        
        self.fusion_32 = FeatureFusionBlock(features)
        self.fusion_16 = FeatureFusionBlock(features)
        self.fusion_8 = FeatureFusionBlock(features)
        self.fusion_4 = FeatureFusionBlock(features)
        
        self.head = nn.Sequential(
            nn.Conv2d(features, features // 2, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(features // 2, 1, kernel_size=1),
            nn.ReLU() 
        )

    def forward(self, x):
        layer_tokens = self.encoder(x)
        t3, t6, t9, t12 = layer_tokens 
        
        f4 = self.reassemble_4(t3)
        f8 = self.reassemble_8(t6)
        f16 = self.reassemble_16(t9)
        f32 = self.reassemble_32(t12)
        
        out = self.fusion_32(f32)
        out = self.fusion_16(f16, previous_stage_output=out)
        out = self.fusion_8(f8, previous_stage_output=out)
        out = self.fusion_4(f4, previous_stage_output=out)
        
        depth_map = self.head(out)
        
        return depth_map

In [11]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

class PretrainedViTEncoder(nn.Module):
    def __init__(self):
        super(PretrainedViTEncoder, self).__init__()
        
        # 1. Download the official, pre-trained ViT-Base (Patch 16, 224x224)
        print("Downloading pre-trained ImageNet weights...")
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        # 2. We don't need the final classification head for depth prediction
        self.vit.heads = nn.Identity()
        
        # 3. Setup hooks to extract the 4 intermediate layers
        self.extracted_features = []
        self._register_hooks()

    def _hook_fn(self, module, input, output):
        # This function runs automatically and saves the output of the layer
        self.extracted_features.append(output)

    def _register_hooks(self):
        # We attach the hook to layers 2, 5, 8, and 11 (which are the 3rd, 6th, 9th, and 12th blocks)
        target_layers = [2, 5, 8, 11]
        for i, layer in enumerate(self.vit.encoder.layers):
            if i in target_layers:
                layer.register_forward_hook(self._hook_fn)

    def forward(self, x):
        # Clear the old features
        self.extracted_features = []
        
        # Run the image through the ViT
        self.vit(x)
        
        # Return the 4 saved layers to the DPT Reassemble block!
        return self.extracted_features

# ==========================================
# How to plug it into your existing code:
# ==========================================

def initialize_production_model(device):
    # 1. Initialize the Pre-Trained Encoder
    encoder = PretrainedViTEncoder()
    
    # Freeze the ViT for the first few epochs! (Optional but highly recommended)
    # This prevents your random decoder weights from sending garbage gradients 
    # back and ruining the perfect ImageNet weights.
    for param in encoder.parameters():
        param.requires_grad = False 
        
    # 2. Plug it into your existing DPT Wrapper
    # Notice we keep embed_dim=768 and spatial_size=14 (14x14 patches)
    model = DPT(vit_encoder=encoder, embed_dim=768, features=256, spatial_size=14)
    model = model.to(device)
    
    print("Production Model Ready with Pre-trained Weights!")
    return model

In [12]:
import os
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.optim import Adam
from tqdm import tqdm

# ---------------------------------------------------------
# 1. The Production-Ready ViT Encoder
# ---------------------------------------------------------
class PretrainedViTEncoder(nn.Module):
    def __init__(self):
        super(PretrainedViTEncoder, self).__init__()
        print("Downloading pre-trained ImageNet weights...")
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        # We do not need the final 1000-class categorization head
        self.vit.heads = nn.Identity() 
        
        self.extracted_features = []
        self._register_hooks()

    def _hook_fn(self, module, input, output):
        self.extracted_features.append(output)

    def _register_hooks(self):
        # Attach to layers 3, 6, 9, and 12 (0-indexed as 2, 5, 8, 11)
        target_layers = [2, 5, 8, 11]
        for i, layer in enumerate(self.vit.encoder.layers):
            if i in target_layers:
                layer.register_forward_hook(self._hook_fn)

    def forward(self, x):
        self.extracted_features = []
        self.vit(x) # Image passes through, and our hooks silently grab the data
        return self.extracted_features

# ---------------------------------------------------------
# 2. The Custom Face Dataset Loader
# ---------------------------------------------------------
class FaceDepthDataset(Dataset):
    def __init__(self, image_dir, depth_dir, transform=None):
        self.image_dir = image_dir
        self.depth_dir = depth_dir
        self.transform = transform
        
        # Get common files to ensure we have perfect pairs
        images_set = set(os.listdir(image_dir))
        depths_set = set(os.listdir(depth_dir))
        
        # Only keep files that exist in both folders
        self.common_files = sorted(list(images_set.intersection(depths_set)))
        
        print(f"✅ Found {len(self.common_files)} perfectly matched pairs.")

    def __len__(self):
        return len(self.common_files)

    def __getitem__(self, idx):
        filename = self.common_files[idx]
        img_path = os.path.join(self.image_dir, filename)
        depth_path = os.path.join(self.depth_dir, filename)

        image = Image.open(img_path).convert("RGB")
        depth = Image.open(depth_path).convert("L")

        if self.transform:
            image = self.transform(image)
            depth = self.transform(depth)

        return image, depth


In [ ]:
def train_dpt():
    # 🔥 UPDATE THESE TWO PATHS 🔥
    IMAGE_DIR = "/kaggle/input/datasets/cantonioupao/prosopo-a-face-dataset-for-3d-reconstruction/prosopo_new/input"
    DEPTH_DIR = "/kaggle/input/datasets/cantonioupao/prosopo-a-face-dataset-for-3d-reconstruction/prosopo_new/depth128x128"
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}\n")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    print("Loading dataset...")
    dataset = FaceDepthDataset(IMAGE_DIR, DEPTH_DIR, transform=transform)
    
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_data, shuffle=True, batch_size=4) 
    test_loader = DataLoader(test_data, shuffle=False, batch_size=4)

    print("\nInitializing DPT Model...")
    encoder = PretrainedViTEncoder()
    
    # Start with Frozen ViT
    for param in encoder.parameters():
        param.requires_grad = False
        
    model = DPT(vit_encoder=encoder).to(device)

    # Hyperparameters
    n_epochs = 30
    unfreeze_epoch = 15
    base_lr = 1e-4 
    
    optimizer = Adam(model.parameters(), lr=base_lr)
    criterion = nn.L1Loss() 

    # Metrics Tracking
    best_test_loss = float('inf')
    history_train_loss = []
    history_test_loss = []

    print("\nStarting Training Pipeline...")
    for epoch in range(n_epochs):
        
        # --- UNFREEZE TRIGGER ---
        if epoch == unfreeze_epoch:
            print("\n🔓 UNFREEZING THE ViT BACKBONE FOR FINE-TUNING!")
            for param in model.encoder.parameters():
                param.requires_grad = True
            for param_group in optimizer.param_groups:
                param_group['lr'] = 1e-5
            print("📉 Learning rate dropped to 1e-5.\n")
        
        # --- TRAIN LOOP ---
        model.train()
        train_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]")
        for batch in loop:
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            y_hat = model(x)
            loss = criterion(y_hat, y)
            train_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        history_train_loss.append(avg_train_loss)
        
        # --- TEST LOOP ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Test] "):
                x, y = batch
                x, y = x.to(device), y.to(device)
                y_hat = model(x)
                loss = criterion(y_hat, y)
                test_loss += loss.item()

        avg_test_loss = test_loss / len(test_loader)
        history_test_loss.append(avg_test_loss)
        
        print(f"--> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

        # --- CHECKPOINT SAVER (ONNX READY) ---
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            print(f"🌟 New best model! Saving state_dict for ONNX export...")
            torch.save(model.state_dict(), "best_dpt_model_onnx_ready.pth")
        print("-" * 50 + "\n")

    # ==============================================================================
    # 5. GENERATE ARTIFACT: TRAINING GRAPH
    # ==============================================================================
    print("Generating Training History Artifact...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, n_epochs + 1), history_train_loss, label='Train Loss (MAE)', color='blue', marker='o')
    plt.plot(range(1, n_epochs + 1), history_test_loss, label='Test Loss (MAE)', color='orange', marker='s')
    
    # Mark the unfreeze epoch
    plt.axvline(x=unfreeze_epoch + 1, color='red', linestyle='--', label='ViT Unfrozen')
    
    plt.title('DPT Face Depth Training History')
    plt.xlabel('Epoch')
    plt.ylabel('L1 Loss (Mean Absolute Error)')
    plt.legend()
    plt.grid(True)
    
    # Save the graph to Kaggle working directory
    plt.savefig('dpt_training_artifact.png', dpi=300, bbox_inches='tight')
    print("✅ Graph saved as 'dpt_training_artifact.png'. Training Complete!")
    plt.show()

train_dpt()